<a href="https://colab.research.google.com/github/fce-laboratoriotp/tp-labo/blob/main/notebook_prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba de Commit Grupal - Laboratorio de Métodos Cuantitativos
Este es un archivo de prueba para verificar la conexión con el repositorio de GitHub y asegurar que los commits de todas las integrantes queden registrados correctamente.

buenasss

In [ ]:
import pandas as pd


hola soy miguelina

hola soy aldi

Hola, soy Kay


hola soy mica

# **Análisis Exploratorio y Detección de Variables**

En esta etapa inicial examinamos la base de datos oficial a nivel provincial (`snic-provincias.csv`) para evaluar su estructura, identificar qué información aporta y verificar los registros del fenómeno en cada jurisdicción.

In [33]:
import pandas as pd

# carga de la base de datos a nivel provincial
df_prov = pd.read_csv("snic-provincias.csv", sep=";", encoding="latin1", on_bad_lines="skip")

# miro el tamaño de la tabla
print("dimensiones de la tabla provincial:", df_prov.shape)

# reviso las primeras filas
display(df_prov.head(3))

# reviso los tipos de datos de cada columna
print("\ntipos de datos:")
print(df_prov.dtypes)

# chequeo si hay valores nulos o celdas vacias
print("\nvalores nulos:")
print(df_prov.isnull().sum())

dimensiones de la tabla provincial: (26528, 14)


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
0,2,Ciudad AutÃ³noma de Buenos Aires,2000,1,Homicidios dolosos,149,158.0,121.0,37.0,0.0,"4,9750013","5,2755051","8,8257008","2,2783601"
1,2,Ciudad AutÃ³noma de Buenos Aires,2000,2,Homicidios dolosos en grado de tentativa,54,58.0,40.0,18.0,0.0,"1,8030207","1,9365778","2,9175868","1,1083914"
2,2,Ciudad AutÃ³noma de Buenos Aires,2000,3,Muertes en accidentes viales,121,126.0,97.0,29.0,0.0,"4,040102","4,2070484","7,0751481","1,7857418"



tipos de datos:
provincia_id                   int64
provincia_nombre              object
anio                           int64
codigo_delito_snic_id         object
codigo_delito_snic_nombre     object
cantidad_hechos                int64
cantidad_victimas            float64
cantidad_victimas_masc       float64
cantidad_victimas_fem        float64
cantidad_victimas_sd         float64
tasa_hechos                   object
tasa_victimas                 object
tasa_victimas_masc            object
tasa_victimas_fem             object
dtype: object

valores nulos:
provincia_id                     0
provincia_nombre                 0
anio                             0
codigo_delito_snic_id            0
codigo_delito_snic_nombre        0
cantidad_hechos                  0
cantidad_victimas            16688
cantidad_victimas_masc       16688
cantidad_victimas_fem        16688
cantidad_victimas_sd         16688
tasa_hechos                      0
tasa_victimas                16688
tasa_victimas_m

In [35]:
# filtramos los nulos para ver a que delitos corresponden
nulos_df = df_prov[df_prov["cantidad_victimas"].isnull()]

# contamos cuantos de esos nulos pertenecen a la categoria de suicidios
suicidios_nulos = nulos_df[nulos_df["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)]

print(f"\ntotal de filas con nulos en todo el dataset: {len(nulos_df)}")
print(f"de esos nulos, cuantos corresponden a suicidios: {len(suicidios_nulos)}")

if len(suicidios_nulos) == 0:
    print("ninguno de los nulos afecta a los suicidios. estan en otros delitos.")
else:
    print("hay registros nulos en suicidios. vamos a ver en que anios ocurren:")
    display(suicidios_nulos[["anio", "codigo_delito_snic_nombre"]])


total de filas con nulos en todo el dataset: 16688
de esos nulos, cuantos corresponden a suicidios: 0
ninguno de los nulos afecta a los suicidios. estan en otros delitos.


### **Extracción y acotamiento del universo de estudio**

El siguiente paso metodológico consiste en extraer exclusivamente los registros asociados al delito de **Suicidios** de la base provincial y acotar el horizonte temporal al quinquenio **2020-2024**, en consonancia con los objetivos del trabajo.

In [34]:
# seleccionamos los registros de suicidios para el periodo de estudio
df_prov_suicidios = df_prov[
    (df_prov["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)) &
    (df_prov["anio"].between(2020, 2024))
].copy()

print(f"cantidad de observaciones seleccionadas para el analisis: {len(df_prov_suicidios)}")
display(df_prov_suicidios.head(3))

cantidad de observaciones seleccionadas para el analisis: 120


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
766,2,Ciudad AutÃ³noma de Buenos Aires,2020,31,Suicidios (consumados),206,206.0,144.0,62.0,0.0,"7,17360364586217","7,17360364586217","10,7500391928512","4,04670947040974"
826,2,Ciudad AutÃ³noma de Buenos Aires,2021,31,Suicidios (consumados),236,236.0,176.0,60.0,0.0,"8,20424395806102","8,20424395806102","13,1013891194452","3,91340674449563"
886,2,Ciudad AutÃ³noma de Buenos Aires,2022,31,Suicidios (consumados),242,242.0,171.0,71.0,0.0,"8,14167866611043","8,14167866611043","10,6373052160119","5,20218931572893"


### **Incorporación y análisis de microdatos**

Para dar cumplimiento a los objetivos del trabajo práctico y acceder a variables demográficas clave (como tramos de edad, sexo e identidad de género), incorporamos la base de **microdatos** (`SAT-SS-BU_2017-2024.csv`). Procedemos a cargarla, inspeccionar su estructura y acotarla al mismo período de estudio.

In [ ]:
# cargamos la base de microdatos
df_micro = pd.read_csv("SAT-SS-BU_2017-2024.csv", sep=",", encoding="latin1", on_bad_lines="skip")

# inspeccionamos dimensiones y columnas
print("dimensiones de la base de microdatos:")
print(f"filas: {df_micro.shape[0]} | columnas: {df_micro.shape[1]}")

print("\ncolumnas disponibles en microdatos:")
print(df_micro.columns.tolist())

# vista preliminar de los registros
display(df_micro.head(3))

# filtramos los microdatos solo para 2020-2024
df_micro_2020_2024 = df_micro[df_micro["anio"].between(2020, 2024)].copy()

print("victimas en microdatos (2020-2024):", len(df_micro_2020_2024))
print("victimas en base provincial (2020-2024):", df_prov_suicidios["cantidad_victimas"].sum())

dimensiones de la base de microdatos:
filas: 30177 | columnas: 27

columnas disponibles en microdatos:
['id_hecho', 'tipo_persona_id', 'federal', 'provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'localidad_id', 'localidad_nombre', 'anio', 'mes', 'fecha_hecho', 'hora_hecho', 'tipo_lugar', 'tipo_lugar_ampliado', 'tipo_lugar_otro', 'modalidad', 'modalidad_ampliado', 'modalidad_otro', 'motivo_origen_registro', 'motivo_origen_registro_otro', 'suicida_sexo', 'suicida_tr_edad', 'suicida_18_aÃ±os_o_mas', 'suicida_clase', 'suicida_clase_otro', 'suicida_identidad_genero']


,id_hecho,tipo_persona_id,federal,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,...,modalidad_ampliado,modalidad_otro,motivo_origen_registro,motivo_origen_registro_otro,suicida_sexo,suicida_tr_edad,suicida_18_aÃ±os_o_mas,suicida_clase,suicida_clase_otro,suicida_identidad_genero
0,8529,Suicida idRegistro 20712,No,42,La Pampa,42002.0,Norte (General Pico),105030,General Pico,2017,...,Ahorcamiento,NaN,IntervenciÃ³n policial,NaN,Masculino,20-24,Si,Sin determinaciÃ³n,NaN,Sin determinar
1,8530,Suicida idRegistro 20713,No,42,La Pampa,42003.0,Sur (General Acha),70030,GuatrachÃ©,2017,...,Arma de fuego,NaN,IntervenciÃ³n policial,NaN,Masculino,55-59,Si,Sin determinaciÃ³n,NaN,Sin determinar
2,8569,Suicida idRegistro 20752,No,78,Santa Cruz,78021.0,GÃ¼er Aike,60,28 de Noviembre,2017,...,Ahorcamiento,NaN,Denuncia particular,NaN,Masculino,15-19,No,Sin determinaciÃ³n,NaN,Sin determinar


victimas en microdatos (2020-2024): 19323
victimas en base provincial (2020-2024): 19323.0


### **Comparación de esquemas entre fuentes**

Una vez filtradas ambas bases para el período **2020-2024**, comparamos sus columnas para identificar qué variables son comunes a ambas fuentes y cuáles son exclusivas de cada una.

In [ ]:
# comparacion de columnas entre la base provincial y la de microdatos
cols_prov = set(df_prov_suicidios.columns)
cols_micro = set(df_micro_2020_2024.columns)

comunes = cols_prov.intersection(cols_micro)
solo_en_prov = cols_prov - cols_micro
solo_en_micro = cols_micro - cols_prov

print(f"columnas comunes ({len(comunes)}):", sorted(list(comunes)))
print(f"columnas exclusivas de provincial ({len(solo_en_prov)}):", sorted(list(solo_en_prov)))
print(f"columnas exclusivas de microdatos ({len(solo_en_micro)}):", sorted(list(solo_en_micro)))

columnas comunes (3): ['anio', 'provincia_id', 'provincia_nombre']
columnas exclusivas de provincial (11): ['cantidad_hechos', 'cantidad_victimas', 'cantidad_victimas_fem', 'cantidad_victimas_masc', 'cantidad_victimas_sd', 'codigo_delito_snic_id', 'codigo_delito_snic_nombre', 'tasa_hechos', 'tasa_victimas', 'tasa_victimas_fem', 'tasa_victimas_masc']
columnas exclusivas de microdatos (24): ['departamento_id', 'departamento_nombre', 'fecha_hecho', 'federal', 'hora_hecho', 'id_hecho', 'localidad_id', 'localidad_nombre', 'mes', 'modalidad', 'modalidad_ampliado', 'modalidad_otro', 'motivo_origen_registro', 'motivo_origen_registro_otro', 'suicida_18_aÃ±os_o_mas', 'suicida_clase', 'suicida_clase_otro', 'suicida_identidad_genero', 'suicida_sexo', 'suicida_tr_edad', 'tipo_lugar', 'tipo_lugar_ampliado', 'tipo_lugar_otro', 'tipo_persona_id']


In [ ]:
import pandas as pd

# =====================================================
# 1) VOLCAR (CARGAR) LAS DOS BASES DE DATOS
# =====================================================
# Cambiar estos nombres por los de sus archivos (o por el link de descarga).
RUTA_1 = "snic-provincias.csv"          # Dataset 1: SNIC por provincia, año y delito
RUTA_2 = "SAT-SS-BU_2017-2024.csv"      # Dataset 2: microdatos, un caso por fila

# Dataset 1: separa con ";" y usa coma como decimal.
# Los códigos se leen como texto (str) para no perder el cero de "02".
df1 = pd.read_csv(
    RUTA_1,
    sep=";",
    decimal=",",
    dtype={"provincia_id": str, "codigo_delito_snic_id": str},
)

# Dataset 2: separa con ",". low_memory=False evita avisos por tipos de dato mezclados.
df2 = pd.read_csv(RUTA_2, dtype={"provincia_id": str}, low_memory=False)

print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)

# =====================================================
# 2) FILTRAR AMBAS POR SUICIDIOS
# =====================================================
# Dataset 1: tiene muchos delitos; nos quedamos con el código 31 = "Suicidios (consumados)".
df1_sui = df1[df1["codigo_delito_snic_id"].str.strip() == "31"].copy()
print("Nombres de delito que quedaron en D1:", df1_sui["codigo_delito_snic_nombre"].unique())

# Dataset 2: cada fila ya es un suicidio, y la columna tipo_persona_id empieza con "Suicida".
# Este filtro funciona como control: si hubiera otra cosa, la saca.
df2_sui = df2[df2["tipo_persona_id"].str.startswith("Suicida", na=False)].copy()
print("Filas D2 antes y después del filtro:", len(df2), len(df2_sui))

# =====================================================
# 3) PREPARAR LAS COLUMNAS PARA UNIRLAS
# =====================================================
# Código de provincia con 2 dígitos en ambos ("2" -> "02"), así coinciden.
df1_sui["provincia_id"] = df1_sui["provincia_id"].str.strip().str.zfill(2)
df2_sui["provincia_id"] = df2_sui["provincia_id"].str.strip().str.zfill(2)

# Sacamos duplicados exactos (en D2, un id_hecho es un hecho).
df1_sui = df1_sui.drop_duplicates()
df2_sui = df2_sui.drop_duplicates(subset="id_hecho")

# fecha_hecho viene como número de Excel (42736 = 01/01/2017): la pasamos a fecha.
df2_sui["fecha_hecho"] = pd.to_datetime(
    pd.to_numeric(df2_sui["fecha_hecho"], errors="coerce"),
    unit="D",
    origin="1899-12-30",
)

# =====================================================
# 4) RESUMIR EL DATASET 2 A NIVEL PROVINCIA + AÑO
# =====================================================
# D1 está por provincia y año; D2 por hecho. Para juntarlos hay que dejar a D2
# en el mismo nivel: contamos cuántos registros hay por provincia y año.
df2_resumen = (
    df2_sui.groupby(["provincia_id", "anio"])
    .agg(
        provincia_nombre_d2=("provincia_nombre", "first"),
        registros_d2=("id_hecho", "count"),
        d2_masculino=("suicida_sexo", lambda s: (s == "Masculino").sum()),
        d2_femenino=("suicida_sexo", lambda s: (s == "Femenino").sum()),
        d2_sexo_sd=("suicida_sexo", lambda s: (~s.isin(["Masculino", "Femenino"])).sum()),
    )
    .reset_index()
)

# =====================================================
# 5) JUNTAR LAS DOS EN UN SOLO DATAFRAME (MERGE)
# =====================================================
# how="outer": no pierde filas de ninguna de las dos bases.
# indicator=True agrega la columna _merge que dice de dónde viene cada fila.
unificado = df1_sui.merge(
    df2_resumen, on=["provincia_id", "anio"], how="outer", indicator=True
)

# Si una provincia está solo en D2, completamos el nombre con el de D2.
unificado["provincia_nombre"] = unificado["provincia_nombre"].fillna(
    unificado["provincia_nombre_d2"]
)
unificado = unificado.drop(columns=["provincia_nombre_d2"])

# Columna más clara que _merge.
unificado["origen"] = unificado["_merge"].map(
    {"both": "ambos", "left_only": "solo_dataset1", "right_only": "solo_dataset2"}
)
unificado = unificado.drop(columns=["_merge"])

# Ordenamos para que se lea mejor.
unificado = unificado.sort_values(["provincia_id", "anio"]).reset_index(drop=True)

# =====================================================
# 6) CONTROLES RÁPIDOS
# =====================================================
print(unificado.shape)
print(unificado["origen"].value_counts())
print("Filas repetidas por provincia+año:",
      unificado.duplicated(subset=["provincia_id", "anio"]).sum())
print(unificado.head())

# =====================================================
# 7) GUARDAR EL ARCHIVO PARA EL GRUPO
# =====================================================
# CSV pensado para Excel en español (";" y coma decimal).
unificado.to_csv("dataset_unificado.csv", index=False,
                 sep=";", decimal=",", encoding="utf-8-sig")
# Excel (si falta: pip install openpyxl)
unificado.to_excel("dataset_unificado.xlsx", index=False)

Dataset 1: (26528, 14)
Dataset 2: (30177, 27)
Nombres de delito que quedaron en D1: ['Suicidios (consumados)']
Filas D2 antes y después del filtro: 30177 30177
(624, 19)
origen
solo_dataset1    432
ambos            192
solo_dataset2      0
Name: count, dtype: int64
Filas repetidas por provincia+año: 0
  provincia_id                 provincia_nombre  anio codigo_delito_snic_id  \
0           02  Ciudad Autónoma de Buenos Aires  2000                    31   
1           02  Ciudad Autónoma de Buenos Aires  2001                    31   
2           02  Ciudad Autónoma de Buenos Aires  2002                    31   
3           02  Ciudad Autónoma de Buenos Aires  2003                    31   
4           02  Ciudad Autónoma de Buenos Aires  2004                    31   

  codigo_delito_snic_nombre  cantidad_hechos  cantidad_victimas  \
0    Suicidios (consumados)              356              356.0   
1    Suicidios (consumados)              317              317.0   
2    Suicidios (consu

In [ ]:
import pandas as pd

# 1) Cargar las dos bases
df1 = pd.read_csv("snic-provincias.csv", sep=";", decimal=",",
                  dtype={"provincia_id": str, "codigo_delito_snic_id": str})
df2 = pd.read_csv("SAT-SS-BU_2017-2024.csv", dtype={"provincia_id": str}, low_memory=False)

# 2) Marcar de qué base viene cada fila
df1["fuente"] = "snic-provincias"
df2["fuente"] = "SAT-SS-BU"

# 3) Juntar una debajo de la otra (las columnas iguales se alinean solas;
#    las que no están en una de las bases quedan vacías)
unificado = pd.concat([df1, df2], ignore_index=True, sort=False)

print(unificado.shape)
print(unificado["fuente"].value_counts())

# 4) Guardar
unificado.to_csv("dataset_unificado.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

# 5) Descargar a tu computadora (en Colab)
from google.colab import files
files.download("dataset_unificado.csv")

(56705, 39)
fuente
SAT-SS-BU          30177
snic-provincias    26528
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>